# High-Level Backtesting of the ICT Strategy

In [ ]:
# imports
import time
import pandas as pd
from nautilus_trader.backtest.config import BacktestVenueConfig, BacktestDataConfig, BacktestRunConfig
from nautilus_trader.backtest.engine import BacktestResult, BacktestEngine, BacktestEngineConfig
from nautilus_trader.backtest.node import BacktestNode
from nautilus_trader.common.config import LoggingConfig
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.model import BarType, Bar, Venue, InstrumentId
from nautilus_trader.model.enums import OmsType
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from nautilus_trader.persistence.config import DataCatalogConfig
from nautilus_trader.test_kit.providers import TestInstrumentProvider
from nautilus_trader.trading import trader
from nautilus_trader.trading.config import ImportableStrategyConfig
import sys
from pathlib import Path

In [ ]:
sys.path.append(str(Path.cwd().parent))

from core.enums import MoneyManagementType

catalog = ParquetDataCatalog("../catalog")

#start_ns = dt_to_unix_nanos(pd.Timestamp("2025-01-01"))
start_ns = dt_to_unix_nanos(pd.Timestamp("2025-07-09"))
end_ns = dt_to_unix_nanos(pd.Timestamp("2025-10-22"))

instrument = TestInstrumentProvider.es_future(2025, 12)
instrument_id = instrument.id.value

# Configure backtesting
venue = BacktestVenueConfig(
    name="GLBX",
    oms_type=OmsType.NETTING,
    account_type="MARGIN",
    base_currency="USD",
    starting_balances=["30_000 USD"],
)

# Configure a catalog for a live system
catalog_cfg = DataCatalogConfig(
    path=str(catalog.path),
    fs_protocol="file",
    name="local"
)

base_bar_type = BarType.from_str(f"{instrument_id}-1-MINUTE-LAST-EXTERNAL")
weekly_bar_type = BarType.from_str(f"{instrument_id}-1-WEEK-LAST-INTERNAL@1-MINUTE-EXTERNAL")
daily_bar_type = BarType.from_str(f"{instrument_id}-1-DAY-LAST-INTERNAL@1-MINUTE-EXTERNAL")

data = BacktestDataConfig(
    catalog_path=str(catalog.path),
    catalog_fs_protocol="file",
    data_cls=Bar,
    bar_types=[base_bar_type],
    instrument_id=instrument_id,
    start_time=start_ns,
    end_time=end_ns
)

engine = BacktestEngineConfig(
    strategies=[
        ImportableStrategyConfig(
            strategy_path="strategies.ict.ict_strategy:ICTStrategy",
            config_path="strategies.ict.ict_strategy:ICTStrategyConfig",
            config={
                "instrument_id": instrument_id,
                "base_bar_type": base_bar_type,
                "weekly_bar_type": weekly_bar_type,
                "daily_bar_type": daily_bar_type,
                "is_backtest": True,

                # ------------- Liquidity Pool Search -------------
                "liquidity_pool_bar_type": BarType.from_str(f"{instrument_id}-1-DAY-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "liquidity_pool_lower_timeframe_bar_type": BarType.from_str(f"{instrument_id}-1-HOUR-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "liquidity_pool_time_delta": pd.Timedelta(hours=24),
                "liquidity_pool_min_lower_timeframe_count": 3,
                "liquidity_pool_extremums_count": 1,
                "liquidity_pool_upper_period_window": 3,
                "liquidity_pool_lower_period_window": 3,

                # ------------- Turtle Soup -------------
                "turtle_soup_analysis_chain_bar_type": [
                    BarType.from_str(f"{instrument_id}-1-HOUR-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                    BarType.from_str(f"{instrument_id}-15-MINUTE-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                    BarType.from_str(f"{instrument_id}-5-MINUTE-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                ],
                "turtle_soup_stop_loss_bar_type": BarType.from_str(f"{instrument_id}-1-HOUR-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "turtle_soup_bars_count": 4,
                "retries_count_on_stop_out": 3,
                "sl_shift": 4.0,

                # ------------- Risk/Reward -------------
                "risk_reward_ratio": 2.0,

                # ------------- Expected Target -------------
                "expected_target_bar_type": BarType.from_str(f"{instrument_id}-1-HOUR-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "expected_target_left": 10,
                "expected_target_right": 10,

                # ------------- Liquidity Pool Reuse -------------
                "liquidity_pool_reuse_bar_type": BarType.from_str(f"{instrument_id}-1-HOUR-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "liquidity_pool_uses_count": 1,

                # ------------- Money Management -------------
                "money_management_type": MoneyManagementType.FIXED_LOT,
                "fixed_lot": 10.0,
                "fixed_risk_percent": 0.5,
            },
        ),
    ],
    logging=LoggingConfig(log_level="ERROR"),
    catalogs=[catalog_cfg]
)

config = BacktestRunConfig(
    engine=engine,
    venues=[venue],
    data=[data],
)

node = BacktestNode(configs=[config])

# run backtesting
elapsed_start = time.perf_counter()
# Runs one or many configs synchronously
results: list[BacktestResult] = node.run()
elapsed_end = time.perf_counter()

print(f"Elapsed time: {elapsed_end - elapsed_start:.6f} seconds")

In [ ]:
result = results[0]

print(f"{'='*60}")
print(f"BACKTEST RESULTS")
print(f"{'='*60}")
print(f"Trader ID:        {result.trader_id}")
print(f"Run ID:           {result.run_id}")
print(f"Backtest Period:  {pd.Timestamp(result.backtest_start, unit='ns')} → {pd.Timestamp(result.backtest_end, unit='ns')}")
print(f"Elapsed Time:     {result.elapsed_time / 1e9:.2f} seconds")
print(f"Iterations:       {result.iterations:,}")
print(f"Total Orders:     {result.total_orders}")
print(f"Total Positions:  {result.total_positions}")

print(f"\n{'='*60}")
print(f"PNL STATISTICS (USD)")
print(f"{'='*60}")
pnl = result.stats_pnls.get('USD', {})
print(f"PnL (total):      ${pnl.get('PnL (total)', 0):,.2f}")
print(f"PnL% (total):     {pnl.get('PnL% (total)', 0):.2f}%")
print(f"Expectancy:       ${pnl.get('Expectancy', 0):,.2f}")
print(f"Win Rate:         {pnl.get('Win Rate', 0)*100:.2f}%")
print(f"Max Winner:       ${pnl.get('Max Winner', 0):,.2f}")
print(f"Avg Winner:       ${pnl.get('Avg Winner', 0):,.2f}")
print(f"Min Winner:       ${pnl.get('Min Winner', 0):,.2f}")
print(f"Max Loser:        ${pnl.get('Max Loser', 0):,.2f}")
print(f"Avg Loser:        ${pnl.get('Avg Loser', 0):,.2f}")
print(f"Min Loser:        ${pnl.get('Min Loser', 0):,.2f}")

print(f"\n{'='*60}")
print(f"RETURN STATISTICS")
print(f"{'='*60}")
ret = result.stats_returns
print(f"Sharpe Ratio (252d):   {ret.get('Sharpe Ratio (252 days)', 0):.4f}")
print(f"Sortino Ratio (252d):  {ret.get('Sortino Ratio (252 days)', 0):.4f}")
print(f"Profit Factor:         {ret.get('Profit Factor', 0):.4f}")
print(f"Returns Volatility:    {ret.get('Returns Volatility (252 days)', 0):.4f}")
print(f"Risk Return Ratio:     {ret.get('Risk Return Ratio', 0):.4f}")
print(f"Avg Return:            {ret.get('Average (Return)', 0)*100:.4f}%")
print(f"Avg Win Return:        {ret.get('Average Win (Return)', 0)*100:.4f}%")
print(f"Avg Loss Return:       {ret.get('Average Loss (Return)', 0)*100:.4f}%")

In [ ]:
backtest_engine: BacktestEngine = node.get_engine(config.id)
positions = backtest_engine.trader.generate_positions_report()

In [ ]:
len(positions)

In [ ]:
pd.set_option("display.max_rows", 202)   # show all rows
pd.set_option("display.max_columns", None)  # show all cols

# Reduce font size for DataFrame display
from IPython.display import display, HTML
display(HTML("<style>.dataframe { font-size: 12px; }</style>"))


In [ ]:
positions

In [ ]:
# Access portfolio analyzer
portfolio = backtest_engine.portfolio
fills_report = backtest_engine.trader.generate_fills_report()

# Get different categories of statistics
stats_pnls = portfolio.analyzer.get_performance_stats_pnls()
stats_returns = portfolio.analyzer.get_performance_stats_returns()
stats_general = portfolio.analyzer.get_performance_stats_general()

In [ ]:
import matplotlib.pyplot as plt


def _money_to_float(series):
    # Convert object/str Money-like series to float values.
    if series.empty:
        return series
    if series.dtype != "O":
        return series
    cleaned = series.astype(str).str.replace(",", "")
    numeric = cleaned.str.split().str[0]
    return pd.to_numeric(numeric, errors="coerce")


positions_report = backtest_engine.trader.generate_positions_report()

if len(positions_report) == 0:
    print("No positions to report.")
else:
    # Cumulative profit in instrument currency
    returns_series = _money_to_float(positions_report["realized_return"])
    returns = returns_series.cumsum()

    # Build equity curve in USD (use realized_pnl if available, otherwise returns)
    starting_balance = 30_000
    if "realized_pnl" in positions_report.columns:
        pnl_series = _money_to_float(positions_report["realized_pnl"])
        equity = pnl_series.cumsum() + starting_balance
    else:
        equity = returns + starting_balance

    peak = equity.cummax()
    drawdown = equity - peak
    drawdown_pct = equity / peak - 1

    print(f"Max drawdown ($): {drawdown.min():,.2f}")
    print(f"Max drawdown (%): {drawdown_pct.min():.2%}")

    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    returns.plot(ax=axes[0], title="Cumulative Returns")
    equity.plot(ax=axes[1], color="navy", title="Equity Curve (USD)")

    drawdown.plot(ax=axes[2], color="firebrick", title="Drawdown ($ / %)")
    axes[2].fill_between(drawdown.index, drawdown, 0, color="firebrick", alpha=0.25, label="Drawdown ($)")

    # Twin axis for percentage drawdown overlay
    ax_dd_pct = axes[2].twinx()
    drawdown_pct.plot(ax=ax_dd_pct, color="darkorange", linestyle="--", label="Drawdown (%)")

    # Combine legends for both axes
    lines, labels = axes[2].get_legend_handles_labels()
    lines2, labels2 = ax_dd_pct.get_legend_handles_labels()
    axes[2].legend(lines + lines2, labels + labels2, loc="lower left")

    plt.tight_layout()
    plt.show()
